# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [ ]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.CIFAR10('cifar10_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [ ]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model.eval()
torch.save(model.state_dict(), 'original_untrained_model.pth')

# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [ ]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output, labels) 
        loss.backward()
        
        # Gradient ascent step
        adv_x = adv_x.detach() + eps_step * adv_x.grad.data.sign()
        
        # Project back to epsilon ball
        delta = adv_x - x
        delta = torch.clamp(delta, min=-eps, max=eps)
        adv_x = torch.clamp(x + delta, min=0, max=1).detach()
   
    return adv_x

In [ ]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
          adv_x.requires_grad_(True)
          model.zero_grad()
          output = model(adv_x)
          batch_size = x.size()[0]
          # TODO: Calculate the loss
          loss = ce_loss(output, labels)
          loss.backward()
          # TODO: compute the adv_x
          # find delta, clamp with eps, project delta to the l2 ball
          # HINT: https://github.com/Harry24k/adversarial-attacks-pytorch/blob/master/torchattacks/attacks/pgdl2.py 
          grad = adv_x.grad.data
          grad_norm = torch.norm(grad.view(batch_size, -1), p=2, dim=1)
          normalized_grad = grad / grad_norm.view(batch_size, 1, 1, 1)
          
          adv_x = adv_x.detach() + eps_step * normalized_grad
          
          delta = adv_x - x
          delta_norm = torch.norm(delta.view(batch_size, -1), p=2, dim=1)
          factor = torch.min(eps / delta_norm, torch.ones_like(delta_norm))
          delta = delta * factor.view(-1, 1, 1, 1)
          adv_x = torch.clamp(x + delta, min=0, max=1).detach()
   
    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [ ]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    tot_test_o, tot_acc_o = 0.0, 0.0
    k = 10
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            # TODO: get x_adv untargeted pgd linf with eps, and eps_step=eps/4
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, k, eps, eps_step=eps/4)
        elif attack == 'pgd_l2':
            # TODO: get x_adv untargeted pgd l2 with eps, and eps_step=eps/4
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, k, eps, eps_step=eps/4)
        else:
            pass
        
        # get the testing accuracy and update tot_test and tot_acc
        with torch.no_grad():
            output = model(x_adv)
            pred = torch.max(output, dim=1)[1]
            tot_acc += (pred == y_batch).sum().item()
            tot_test += y_batch.size(0)
            
            output_o = model(x_batch)
            pred_o = torch.max(output_o, dim=1)[1]
            tot_acc_o += (pred_o == y_batch).sum().item()
            tot_test_o += y_batch.size(0)  
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')
    print('Standard accuracy %.5lf' % (tot_acc_o/tot_test_o), f'on original attack with eps = {eps}')

## Single-Norm Robust Accuracy

In [ ]:
'''
# Evaluate on Linf attack with different models with eps = 8/255
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 
'''

In [ ]:
'''
# Evaluate on L2 attack with different models with eps = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 
'''

## Multi-Norm Robust Accuracy

In [ ]:
def test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    tot_test_o, tot_acc_o = 0.0, 0.0
    k = 10
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # TODO: get x_adv_linf and x_adv_l2 untargeted pgd linf and l2 with eps, and eps_step=eps/4
        x_adv_linf = pgd_linf_untargeted(model, x_batch, y_batch, k, eps_linf, eps_step=eps_linf/4)
        x_adv_l2 = pgd_l2_untargeted(model, x_batch, y_batch, k, eps_l2, eps_step = eps_l2/4)
        
        ## calculate union accuracy: correct only if both attacks are correct
        
        out = model(x_adv_linf)
        pred_linf = torch.max(out, dim=1)[1]
        out = model(x_adv_l2)
        pred_l2 = torch.max(out, dim=1)[1]
        
        # TODO: get the testing accuracy with multi-norm robustness and update tot_test and tot_acc
        tot_acc += ((pred_linf == y_batch) & (pred_l2 == y_batch)).sum().item()
        tot_test += y_batch.size(0)
        
        output_o = model(x_batch)
        pred_o = torch.max(output_o, dim=1)[1]
        tot_acc_o += (pred_o == y_batch).sum().item()
        tot_test_o += y_batch.size(0)  
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on multi attacks')
    print('Standard accuracy %.5lf' % (tot_acc_o/tot_test_o), f'on original attack')

In [ ]:
''' 
# Evaluate on multi-norm attacks with different models with eps_linf = 8./255, eps_l2 = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on multi attacks with model 1
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on multi attacks with model 2
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on multi attacks with model 3
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)
'''


Standard Accuracy Evaluation Function

In [ ]:
def evaluate_standard_accuracy(model, test_loader, device):
    """Evaluate standard accuracy on clean test data"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

In [ ]:
def evaluate_robust_accuracy(model, test_loader, device, eps, k=10, eps_step=None):
    """Evaluate robust accuracy against PGD attack"""
    # CRITICAL: For FGSM (k=1), eps_step MUST equal eps (one full step)
    # For PGD (k>1), use smaller steps (eps/4)
    if eps_step is None:
        if k == 1:
            eps_step = eps  # FGSM: ONE full epsilon step
        else:
            eps_step = eps / 4  # PGD: multiple smaller steps
    
    model.eval()
    correct = 0
    total = 0
    
    print(f"  Attack config: k={k}, eps={eps:.4f}, eps_step={eps_step:.4f}")
    
    for inputs, targets in tqdm(test_loader, desc=f'Attacking (k={k})'):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Generate adversarial examples
        adv_inputs = pgd_linf_untargeted(model, inputs, targets, k, eps, eps_step)
        
        # Evaluate on adversarial examples
        with torch.no_grad():
            outputs = model(adv_inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

In [ ]:
def adversarial_training(model, train_loader, test_loader, device, 
                        epochs=10, lr=0.1, eps=8/255, k=7, eps_step=None):
    """
    PGD-based Adversarial Training
    
    Key: Train on adversarial examples (adv_x) with correct labels (y_batch)
    
    Args:
        model: Neural network model
        train_loader: Training data loader
        test_loader: Test data loader
        device: cuda or cpu
        epochs: Number of training epochs
        lr: Learning rate
        eps: Epsilon for PGD attack (perturbation budget)
        k: Number of PGD steps during training
        eps_step: Step size for each PGD iteration
    """
    if eps_step is None:
        eps_step = eps / 4
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, 
                                               milestones=[int(epochs*0.5), int(epochs*0.75)], 
                                               gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'standard_acc': [],
        'robust_acc': []
    }
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs, targets = inputs.to(device), targets.to(device)
            model.eval()
            adv_inputs = pgd_linf_untargeted(model, inputs, targets, k, eps, eps_step)
            
            model.train()
            
            optimizer.zero_grad()
            outputs = model(adv_inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'Loss': f'{train_loss/(batch_idx+1):.3f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
        
        scheduler.step()
        
        # Record training metrics
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100. * correct / total)
        
        # Evaluate after each epoch
        print(f'\nEpoch {epoch+1} Results:')
        standard_acc = evaluate_standard_accuracy(model, test_loader, device)
        robust_acc = evaluate_robust_accuracy(model, test_loader, device, eps, k=10, eps_step=eps/4)
        
        history['standard_acc'].append(standard_acc)
        history['robust_acc'].append(robust_acc)
        
        print(f'Standard Accuracy: {standard_acc:.2f}%')
        print(f'Robust Accuracy (eps={eps:.4f}): {robust_acc:.2f}%')
        print('-' * 60)
    
    return model, history

In [ ]:
# Part a
epsilon_values = [0.01, 0.05, 0.1]
all_results = {}

print(f"Testing {len(epsilon_values)} epsilon values: {epsilon_values}")
print(f"Using k={10} PGD steps during training\n")

for eps in epsilon_values:
    print(f"\n{'='*80}")
    print(f"Training with epsilon = {eps:.4f}")
    print(f"{'='*80}\n")
    
    # Initialize new model for each epsilon
    model_adv = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
    
    # Train model with adversarial training
    trained_model, history = adversarial_training(
        model=model_adv,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=10,  # Increase to 50-100 for better results
        lr=0.1,
        eps=eps,
        k=10,  # Using k=10 PGD steps
        eps_step=eps/4
    )
    
    # Save the trained model with epsilon value in filename
    model_name = f'adv_trained_eps_{eps:.4f}.pth'
    torch.save(trained_model.state_dict(), model_name)
    print(f"\nModel saved as: {model_name}")
    
    # Store results with epsilon as key
    all_results[f'eps_{eps:.4f}'] = history
    
    # Final evaluation
    print(f"\n{'='*80}")
    print(f"Final Results for eps={eps:.4f}:")
    print(f"Standard Accuracy: {history['standard_acc'][-1]:.2f}%")
    print(f"Robust Accuracy: {history['robust_acc'][-1]:.2f}%")
    print(f"Accuracy Drop: {history['standard_acc'][-1] - history['robust_acc'][-1]:.2f}%")
    print(f"{'='*80}\n")

print("\n" + "="*80)
print("TRAINING COMPLETE FOR ALL EPSILON VALUES")
print("="*80)

In [ ]:
# ============================================================================
# DIAGNOSTIC: Verify FGSM is working correctly
# ============================================================================
print("="*80)
print("DIAGNOSTIC: Testing FGSM Implementation")
print("="*80)

# Test on a small batch manually
test_batch = next(iter(test_loader))
x_test, y_test = test_batch[0].to(device), test_batch[1].to(device)

# Test epsilon
test_eps = 0.05

print(f"\nTesting on one batch with ε={test_eps}")
print(f"Batch size: {x_test.size(0)}")

# Get clean predictions
standard_model.eval()
with torch.no_grad():
    clean_out = standard_model(x_test)
    clean_pred = torch.max(clean_out, dim=1)[1]
    clean_acc = (clean_pred == y_test).sum().item() / y_test.size(0)

print(f"Clean accuracy on batch: {clean_acc*100:.2f}%")

# Test FGSM manually with k=1, eps_step=eps
print(f"\nGenerating FGSM attack (k=1, eps_step={test_eps})...")
adv_x_fgsm = pgd_linf_untargeted(standard_model, x_test, y_test, k=1, eps=test_eps, eps_step=test_eps)

with torch.no_grad():
    adv_out = standard_model(adv_x_fgsm)
    adv_pred = torch.max(adv_out, dim=1)[1]
    adv_acc = (adv_pred == y_test).sum().item() / y_test.size(0)

print(f"FGSM accuracy on batch: {adv_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - adv_acc)*100:.2f}%")

# Check perturbation magnitude
perturbation = (adv_x_fgsm - x_test).abs()
print(f"\nPerturbation statistics:")
print(f"  Max perturbation: {perturbation.max().item():.4f}")
print(f"  Mean perturbation: {perturbation.mean().item():.4f}")
print(f"  Expected max (epsilon): {test_eps:.4f}")

if abs(perturbation.max().item() - test_eps) < 0.001:
    print("✓ Perturbation magnitude looks correct!")
else:
    print("✗ WARNING: Perturbation is not at epsilon boundary!")

# Compare with PGD-10
print(f"\nComparing with PGD-10...")
adv_x_pgd10 = pgd_linf_untargeted(standard_model, x_test, y_test, k=10, eps=test_eps, eps_step=test_eps/4)

with torch.no_grad():
    pgd_out = standard_model(adv_x_pgd10)
    pgd_pred = torch.max(pgd_out, dim=1)[1]
    pgd_acc = (pgd_pred == y_test).sum().item() / y_test.size(0)

print(f"PGD-10 accuracy on batch: {pgd_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - pgd_acc)*100:.2f}%")

if pgd_acc < adv_acc:
    print("✓ PGD-10 is stronger than FGSM (as expected)")
else:
    print("✗ WARNING: FGSM should be weaker than PGD-10!")

print("\n" + "="*80)
print("If you see ✗ warnings above, your FGSM implementation is incorrect!")
print("="*80 + "\n")

In [ ]:
# Train a standard (non-adversarial) model for comparison in Part (b)
print("Training standard model for comparison...")
standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
optimizer = optim.SGD(standard_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

# Standard training
num_epochs = 10
for epoch in range(num_epochs):
    standard_model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Standard Training {epoch+1}/{num_epochs}')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = standard_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'Loss': f'{train_loss/(len(pbar)):.3f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
    
    # Evaluate
    std_acc = evaluate_standard_accuracy(standard_model, test_loader, device)
    print(f'Epoch {epoch+1}: Standard Accuracy = {std_acc:.2f}%')

torch.save(standard_model.state_dict(), 'standard_trained.pth')
print("\nStandard model training complete!")
print(f"Final Standard Accuracy: {evaluate_standard_accuracy(standard_model, test_loader, device):.2f}%")

In [ ]:
# Part (b) - Load RAMP pretrained model and compare with adversarially trained model

import os
import torch

print("="*80)
print("Loading Models for Part (b) Comparison")
print("="*80)

# Load the RAMP pretrained model as the "standard" baseline
print("\nLoading RAMP pretrained model (standard non-robust baseline)...")
standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
standard_model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
standard_model.eval()
print("✓ Loaded RAMP pretrained model")

# Load your adversarially trained model from Part (a)
test_eps = 0.05  # The epsilon you used for adversarial training

print(f"\nLoading your adversarially trained model (ε={test_eps})...")
adv_trained_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
adv_trained_model.load_state_dict(torch.load(f'adv_trained_eps_{test_eps:.4f}.pth'))
adv_trained_model.eval()
print(f"✓ Loaded your adversarially trained model (ε={test_eps:.4f})")

# Quick evaluation of both models
print("\n" + "-"*80)
print("Quick Evaluation:")
print("-"*80)
print("Evaluating RAMP model clean accuracy...")
standard_clean = evaluate_standard_accuracy(standard_model, test_loader, device)
print(f"RAMP Model Clean Accuracy: {standard_clean:.2f}%")

print("\nEvaluating your adversarially trained model...")
adv_clean = evaluate_standard_accuracy(adv_trained_model, test_loader, device)
print(f"Your Model Clean Accuracy: {adv_clean:.2f}%")

print("\n" + "="*80)
print("Models loaded successfully! Ready for Part (b) comparison.")
print("="*80)
print(f"\nComparison setup:")
print(f"  Baseline: RAMP pretrained model (standard non-robust)")
print(f"  Your model: Adversarially trained with ε={test_eps}")
print(f"\nNow run the comparison function to test FGSM attacks on both models.\n")

In [ ]:
# Complete Part (b) - Attack Comparison Script

import torch
import matplotlib.pyplot as plt
import numpy as np

# ============================================================================
# Step 1: Load Models
# ============================================================================
print("="*80)
print("PART (b): Comparing Attack Effectiveness")
print("="*80)

# Load RAMP pretrained model (standard baseline)
print("\n[1/3] Loading RAMP pretrained model...")
standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
standard_model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
standard_model.eval()
print("✓ RAMP model loaded")

# Load your adversarially trained model
test_eps = 0.05  # The epsilon you used for adversarial training
print(f"\n[2/3] Loading your adversarially trained model (ε={test_eps})...")
adv_trained_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
adv_trained_model.load_state_dict(torch.load(f'adv_trained_eps_{test_eps:.4f}.pth'))
adv_trained_model.eval()
print(f"✓ Your model loaded")

# ============================================================================
# Step 2: Run Comparison
# ============================================================================
print("\n[3/3] Running attack comparison...")
print("="*80)

# Test epsilon values (matching Part a)
epsilon_values = [0.01, 0.05, 0.1]

# Store results
results = {
    'epsilons': epsilon_values,
    'standard': {'clean': 0, 'robust': []},
    'adv_trained': {'clean': 0, 'robust': []}
}

# Evaluate clean accuracy
print("\n" + "="*80)
print("Evaluating Clean Accuracy (No Attack)")
print("="*80)
results['standard']['clean'] = evaluate_standard_accuracy(standard_model, test_loader, device)
results['adv_trained']['clean'] = evaluate_standard_accuracy(adv_trained_model, test_loader, device)

print(f"RAMP Model:          {results['standard']['clean']:.2f}%")
print(f"Your Trained Model:  {results['adv_trained']['clean']:.2f}%")
clean_diff = results['standard']['clean'] - results['adv_trained']['clean']
print(f"Clean Accuracy Cost: {clean_diff:.2f}% (trade-off for robustness)")

# Test FGSM attacks at different epsilon values
print("\n" + "="*80)
print("Testing FGSM Attack (1-step PGD) at Different Epsilons")
print("="*80)

for i, eps in enumerate(epsilon_values, 1):
    print(f"\n[Attack {i}/{len(epsilon_values)}] Epsilon = {eps:.4f}")
    print("-"*80)
    
    # Attack RAMP model
    print("Attacking RAMP model with FGSM...")
    std_robust = evaluate_robust_accuracy(
        standard_model, test_loader, device,
        eps=eps, k=1  # FGSM: k=1, eps_step will be set to eps automatically
    )
    results['standard']['robust'].append(std_robust)
    
    # Attack your model
    print("Attacking your adversarially trained model with FGSM...")
    adv_robust = evaluate_robust_accuracy(
        adv_trained_model, test_loader, device,
        eps=eps, k=1  # FGSM: k=1, eps_step will be set to eps automatically
    )
    results['adv_trained']['robust'].append(adv_robust)
    
    # Calculate metrics
    std_drop = results['standard']['clean'] - std_robust
    adv_drop = results['adv_trained']['clean'] - adv_robust
    improvement = adv_robust - std_robust
    
    # Print results
    print(f"\nResults at ε={eps:.4f}:")
    print(f"  RAMP Model:         {std_robust:.2f}% (↓{std_drop:.2f}%)")
    print(f"  Your Trained Model: {adv_robust:.2f}% (↓{adv_drop:.2f}%)")
    print(f"  Robustness Gain:    +{improvement:.2f}%")
    
    if std_drop > 0.1 and adv_drop > 0.1:
        effectiveness = std_drop / adv_drop
        print(f"  Attack Effectiveness Ratio: {effectiveness:.2f}x")
        print(f"    → Attack is {effectiveness:.2f}x MORE effective on RAMP model")
    elif std_drop < 0.1:
        print(f"  ⚠️  RAMP model shows minimal drop (already weak)")

# ============================================================================
# Step 3: Print Summary and Visualization
# ============================================================================
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"\nClean Accuracy:")
print(f"  RAMP:       {results['standard']['clean']:.2f}%")
print(f"  Your Model: {results['adv_trained']['clean']:.2f}%")

print(f"\nRobust Accuracy under FGSM Attack:")
print(f"{'Epsilon':<12} {'RAMP':<12} {'Your Model':<12} {'Improvement':<12}")
print("-"*50)
for i, eps in enumerate(epsilon_values):
    std_acc = results['standard']['robust'][i]
    adv_acc = results['adv_trained']['robust'][i]
    improvement = adv_acc - std_acc
    print(f"{eps:<12.4f} {std_acc:<12.2f} {adv_acc:<12.2f} +{improvement:<11.2f}")

# Plot results
print("\n" + "="*80)
print("Generating Visualization...")
print("="*80)

plt.figure(figsize=(12, 5))

# Plot 1: Accuracy under attack
plt.subplot(1, 2, 1)
plt.plot(epsilon_values, [results['standard']['clean']]*len(epsilon_values), 
         'b--', label='RAMP (Clean)', alpha=0.5)
plt.plot(epsilon_values, results['standard']['robust'], 
         'b-o', label='RAMP (Under Attack)', linewidth=2)
plt.plot(epsilon_values, [results['adv_trained']['clean']]*len(epsilon_values), 
         'r--', label='Your Model (Clean)', alpha=0.5)
plt.plot(epsilon_values, results['adv_trained']['robust'], 
         'r-s', label='Your Model (Under Attack)', linewidth=2)
plt.xlabel('Epsilon (ε)', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Model Accuracy Under FGSM Attack', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy drop
plt.subplot(1, 2, 2)
std_drops = [results['standard']['clean'] - acc for acc in results['standard']['robust']]
adv_drops = [results['adv_trained']['clean'] - acc for acc in results['adv_trained']['robust']]
x = np.arange(len(epsilon_values))
width = 0.35
plt.bar(x - width/2, std_drops, width, label='RAMP Model', color='blue', alpha=0.7)
plt.bar(x + width/2, adv_drops, width, label='Your Model', color='red', alpha=0.7)
plt.xlabel('Epsilon (ε)', fontsize=12)
plt.ylabel('Accuracy Drop (%)', fontsize=12)
plt.title('Attack Impact (Accuracy Drop)', fontsize=14, fontweight='bold')
plt.xticks(x, [f'{eps:.2f}' for eps in epsilon_values])
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('part_b_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved as 'part_b_comparison.png'")
plt.show()

print("\n" + "="*80)
print("Part (b) Comparison Complete!")
print("="*80)

In [ ]:
# Quick FGSM verification
print("Testing FGSM implementation...")
test_eps = 0.05

# Test with k=1 (should be FGSM)
fgsm_result = evaluate_robust_accuracy(standard_model, test_loader, device, eps=test_eps, k=1)
print(f"FGSM (k=1) result: {fgsm_result:.2f}%")

# Test with k=10 (should be PGD)
pgd_result = evaluate_robust_accuracy(standard_model, test_loader, device, eps=test_eps, k=10)
print(f"PGD-10 result: {pgd_result:.2f}%")

print(f"\nExpected: PGD-10 should be MORE effective (lower accuracy) than FGSM")
print(f"Actual: FGSM={fgsm_result:.2f}%, PGD-10={pgd_result:.2f}%")
if pgd_result < fgsm_result:
    print("✓ Correct! PGD is more effective than FGSM")
else:
    print("✗ Something is still wrong")

In [ ]:
# ============================================================================
# EMERGENCY DIAGNOSTIC - What's actually happening in the attack?
# ============================================================================
print("="*80)
print("EMERGENCY DIAGNOSTIC")
print("="*80)

# Manually test FGSM on one batch
test_batch = next(iter(test_loader))
x_test, y_test = test_batch[0][:10].to(device), test_batch[1][:10].to(device)

print(f"\nTesting FGSM attack manually...")
print(f"Epsilon: 0.05")
print(f"Expected: k=1, eps_step=0.05 (one full step)")

# Clean accuracy
standard_model.eval()
with torch.no_grad():
    clean_out = standard_model(x_test)
    clean_pred = torch.max(clean_out, dim=1)[1]
    clean_correct = (clean_pred == y_test).sum().item()

print(f"\nClean: {clean_correct}/10 correct")

# Method 1: Your current pgd_linf_untargeted
print("\n[Method 1] Using pgd_linf_untargeted(k=1, eps=0.05, eps_step=0.05):")
adv_x_1 = pgd_linf_untargeted(standard_model, x_test, y_test, k=1, eps=0.05, eps_step=0.05)
with torch.no_grad():
    out_1 = standard_model(adv_x_1)
    pred_1 = torch.max(out_1, dim=1)[1]
    correct_1 = (pred_1 == y_test).sum().item()
print(f"  Result: {correct_1}/10 correct")
print(f"  Max perturbation: {(adv_x_1 - x_test).abs().max().item():.6f}")

# Method 2: Manual FGSM (ground truth)
print("\n[Method 2] Manual FGSM implementation:")
x_adv_manual = x_test.clone().detach()
x_adv_manual.requires_grad = True

standard_model.zero_grad()
output = standard_model(x_adv_manual)
loss = torch.nn.CrossEntropyLoss()(output, y_test)
loss.backward()

# ONE step with FULL epsilon
with torch.no_grad():
    x_adv_manual = x_adv_manual + 0.05 * x_adv_manual.grad.sign()
    # Clip to epsilon ball
    perturbation = torch.clamp(x_adv_manual - x_test, min=-0.05, max=0.05)
    x_adv_manual = torch.clamp(x_test + perturbation, min=0, max=1)

with torch.no_grad():
    out_manual = standard_model(x_adv_manual)
    pred_manual = torch.max(out_manual, dim=1)[1]
    correct_manual = (pred_manual == y_test).sum().item()

print(f"  Result: {correct_manual}/10 correct")
print(f"  Max perturbation: {(x_adv_manual - x_test).abs().max().item():.6f}")

# Compare
print("\n" + "="*80)
if correct_1 == correct_manual:
    print("✓ Methods match! Your pgd_linf_untargeted is working correctly.")
    print("  Problem must be in evaluate_robust_accuracy function.")
else:
    print("✗ Methods DON'T match! Your pgd_linf_untargeted is broken.")
    print(f"  Your function: {correct_1}/10 correct")
    print(f"  Manual FGSM: {correct_manual}/10 correct")

print("="*80)